In [ ]:
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

# ===== Configuration =====
PINECONE_API_KEY = "pcsk_4DqA35_BYaWgQCoVUkGTNDRYenf3NQzwzZX6C685nC2fwMj5qXgnpMXmUcH1eVXRjVfMgw"
PINECONE_ENVIRONMENT = "us-east-1"
PINECONE_INDEX_NAME = "quickstart-new-test-csv-html-merged-final"
VECTOR_DIM = 384

CSV_FOLDER_PATH = "../data/cleaned"

CSV_TO_HTML  = {
    "bpx_i_cleaned.csv": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/BPX_I.htm",
    "demo_i_cleaned.csv": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DEMO_I.htm",
    "diq_i_cleaned.csv": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DIQ_I.htm",
    "ghb_i_cleaned.csv": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/GHB_I.htm",
    "ins_i_cleaned.csv": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/INS_I.htm",
    "ogtt_i_cleaned.csv": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/OGTT_I.htm"
}


def init_pinecone():
    pc = Pinecone(api_key=PINECONE_API_KEY)
    return pc.Index(PINECONE_INDEX_NAME)

def parse_html_variable_descriptions(url):
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    variables_info = {}

    # Try parsing table format (common in NHANES documentation)
    table = soup.find("table", class_="DataTable")
    if table:
        for row in table.find_all("tr")[1:]:  # Skip header
            cols = row.find_all("td")
            if len(cols) >= 2:
                var_name = cols[0].text.strip()
                description = cols[1].text.strip()
                variables_info[var_name] = description
    else:
        # fallback to previous logic
        for header in soup.find_all(['h3', 'h4']):
            var_name = header.text.strip()
            desc_tag = header.find_next_sibling()
            if desc_tag:
                variables_info[var_name] = desc_tag.text.strip()

    return variables_info


def upsert_html_descriptions(index, html_map):
    model = SentenceTransformer("all-MiniLM-L6-v2")

    all_vectors = []
    for csv_file, html_url in html_map.items():
        print(f"Fetching HTML from: {html_url}")
        var_descs = parse_html_variable_descriptions(html_url)

        for var, desc in var_descs.items():
            text = f"{var}: {desc}"
            embedding = model.encode(text)
            vector_id = f"{csv_file}|{var}|desc"
            metadata = {
                "text": text,
                "file": csv_file,
                "variable": var,
                "type": "variable_doc"
            }
            all_vectors.append((vector_id, embedding.tolist(), metadata))
            print(f"Prepared vector: {vector_id}")

    print(f"Upserting {len(all_vectors)} variable descriptions to Pinecone...")
    index.upsert(vectors=all_vectors)

def main():
    index = init_pinecone()
    upsert_html_descriptions(index, CSV_TO_HTML)

if __name__ == "__main__":
    main()
